# Benchmark Construction Pipeline

In [41]:
import json, math, re, random
import pandas as pd, numpy as np
from collections import defaultdict
from dotenv import load_dotenv
from openai import OpenAI
import time
from pathlib import Path
import itertools
from tools import filter_data, exclude, count_by, sentiment_breakdown, add_shares, add_priority, apply_min_volume, rank_top, share_of, two_prop_test, chi_squared

pd.set_option("display.max_colwidth", None)   # None = don't truncate cell values (show full column text)
random.seed(29)

In [42]:
OUT_DIR = Path("../benchmark_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [43]:
# llm setup
load_dotenv()
client = OpenAI()
MODEL = "vertex_ai/gemini-2.5-flash" 

In [44]:
# cost logging
PRICE_IN, PRICE_OUT = 0.30/1e6, 2.50/1e6     # USD/token — verify current gemini-flash pricing
USAGE = defaultdict(lambda: {"calls":0, "in":0, "out":0})
USAGE.clear()


def gemini(prompt, block, max_tokens=2000, response_format=None):
    kwargs = dict(model=MODEL,
                  messages=[{"role": "user", "content": prompt}],
                  max_tokens=max_tokens)
    if response_format:
        kwargs["response_format"] = response_format
    r = client.chat.completions.create(**kwargs)
    u = r.usage
    USAGE[block]["calls"] += 1
    USAGE[block]["in"] += u.prompt_tokens
    USAGE[block]["out"] += u.completion_tokens
    choice = r.choices[0]
    if getattr(choice, "finish_reason", None) == "length":
        raise ValueError(f"[{block}] response truncated at max_tokens={max_tokens} "
                         f"(generated {u.completion_tokens} tokens) -- raise max_tokens or batch the call")
    return choice.message.content


# helper - call gemini, strip json fences, parse JSON

def _extract_json_payload(text):
    if isinstance(text, (dict, list)):
        return text
    if text is None:
        raise ValueError("empty response")
    if not isinstance(text, str):
        text = str(text)

    cleaned = text.strip()
    if not cleaned:
        raise ValueError("empty response")

    # Remove fenced code blocks even if the model adds a language tag.
    cleaned = re.sub(r"^```(?:json)?\s*", "", cleaned, flags=re.IGNORECASE | re.MULTILINE)
    cleaned = re.sub(r"\s*```$", "", cleaned, flags=re.IGNORECASE | re.MULTILINE).strip()

    # Normalise smart SINGLE quotes only (apostrophes). Do NOT touch smart double quotes: they are
    # JSON string delimiters, so rewriting a curly double-quote inside answer text to a straight one
    # turns a valid payload into an unescaped-quote parse error.
    cleaned = cleaned.replace("‘", "'").replace("’", "'")

    # strict=False tolerates raw newlines/tabs the model may leave inside long answer strings.
    try:
        return json.loads(cleaned, strict=False)
    except json.JSONDecodeError:
        pass

    decoder = json.JSONDecoder(strict=False)
    for start in range(len(cleaned)):
        if cleaned[start] not in "{[":
            continue
        try:
            obj, end = decoder.raw_decode(cleaned[start:])
        except json.JSONDecodeError:
            continue
        if end > 0:
            return obj

    # show head AND tail: a clean head with a cut-off tail means the response was truncated.
    head = cleaned[:150].replace("\n", " ")
    tail = cleaned[-150:].replace("\n", " ")
    raise ValueError(f"could not parse JSON payload ({len(cleaned)} chars) -- if the tail looks "
                     f"cut off the response was truncated. head: {head!r} ... tail: {tail!r}")


def gemini_json(prompt, block, max_tokens=6000, retries=3):
    last_error = None
    for attempt in range(retries):
        txt = gemini(
            prompt,
            block,
            max_tokens,
            response_format={"type": "json_object"},
        )
        try:
            return _extract_json_payload(txt)
        except ValueError as exc:
            last_error = exc
            if attempt < retries - 1:
                continue
    raise ValueError(f"[{block}] invalid JSON response after {retries} attempts: {last_error}") from last_error


def cost_report():
    tot=0
    for b,d in USAGE.items():
        c=d["in"]*PRICE_IN+d["out"]*PRICE_OUT; tot+=c
        print(f"{b:12} calls={d['calls']:3d}  in={d['in']:7d}  out={d['out']:7d}  ${c:.4f}")
    print(f"{'TOTAL':12} {'':22} ${tot:.4f}")

# Step 1: load the generated dataset

Built by `data_generation.ipynb` and frozen under `benchmark_outputs/{VERSION}/`. Switch `MODE` to
build the task set against the other dataset - everything downstream reads `CORPUS`.

- `labels.csv` - one row per review; the only file an agent ever sees.
- `generation_config.yaml` - the industries, aspects and org names the task templates fill in.
- `cells.csv` is deliberately **not** loaded here: it holds the planted ground truth, and nothing in
  the task pipeline is allowed to see it.

In [ ]:
import yaml

VERSION, MODE = "v1", "hard"          # "easy" | "hard"
VER_DIR = OUT_DIR / VERSION
CONFIG  = yaml.safe_load((VER_DIR / "generation_config.yaml").read_text())
INDUSTRIES, ASPECTS, ORGS = CONFIG["industries"], CONFIG["aspects"], CONFIG["orgs"]
CORPUS  = pd.read_csv(VER_DIR / MODE / "labels.csv")

print(f"{VERSION}/{MODE}: {len(CORPUS):,} reviews | {len(INDUSTRIES)} industries x "
      f"{len(ASPECTS)} aspects x {sum(len(v) for v in ORGS.values())} orgs "
      f"| {CONFIG['unanswerable'][MODE]:.1%} of questions unanswerable")
CORPUS.head()

# Step 2: define tasks

Tasks are built against `CORPUS`, i.e. whichever mode Step 1 generated. Re-run Step 1 with the
other `MODE` to get that mode's task set.

## Build solver functions

In [17]:
# build solver functions for each task.
# each solver runs the tools on the corpus to produce the gold answer + records the tool calls.
# `where` selects a segment: {"industry": name} or {"org": name}. filter is implicit (not a step).

# ---- 1-step solvers (descriptive) ----

def solve_count(where, asp, sent):
    # share_of returns {n, count, share}; read the count for a "how many" answer
    r = share_of(filter_data(CORPUS, **where, child_aspect=asp), sent)
    return r["count"], [
        {"tool":"share_of","args":{**where,"child_aspect":asp,"sentiment":sent}}]

def solve_share(where, asp, sent):
    r = share_of(filter_data(CORPUS, **where, child_aspect=asp), sent)
    return r["share"], [
        {"tool":"share_of","args":{**where,"child_aspect":asp,"sentiment":sent}}]

# ---- 2-step solvers ----

def solve_top_k(where, k, by):
    filt = {**where, **({"sentiment":"negative"} if by=="negative" else {})}
    top = rank_top(count_by(filter_data(CORPUS, **filt), "child_aspect"), by="count", top_n=k)
    return top.child_aspect.tolist(), [
        {"tool":"count_by","args":{**filt,"group_by":"child_aspect"}},
        {"tool":"rank_top","args":{"by":"count","top_n":k}}]

def solve_split(where, group_by):
    sh = add_shares(sentiment_breakdown(filter_data(CORPUS, **where), group_by))
    v = {k: float(sh[k].iloc[0]) for k in ["pos_share","neg_share","neu_share"]}
    return v, [
        {"tool":"sentiment_breakdown","args":{**where,"group_by":group_by}},
        {"tool":"add_shares","args":{}}]

def solve_compare(where_a, where_b, asp, sent):
    ra = share_of(filter_data(CORPUS, **where_a, child_aspect=asp), sent)
    rb = share_of(filter_data(CORPUS, **where_b, child_aspect=asp), sent)
    return {"a":ra["share"],"b":rb["share"],"higher":"a" if (ra["share"] or 0)>(rb["share"] or 0) else "b"}, [
        {"tool":"share_of","args":{**where_a,"child_aspect":asp,"sentiment":sent}},
        {"tool":"share_of","args":{**where_b,"child_aspect":asp,"sentiment":sent}}]

# ---- 3-step solver ----

def solve_two_prop(where_a, where_b, asp, sent):
    da = filter_data(CORPUS, **where_a, child_aspect=asp)
    db = filter_data(CORPUS, **where_b, child_aspect=asp)
    r = two_prop_test(da, db, sent)
    return r, [
        {"tool":"share_of","args":{**where_a,"child_aspect":asp,"sentiment":sent}},
        {"tool":"share_of","args":{**where_b,"child_aspect":asp,"sentiment":sent}},
        {"tool":"two_prop_test","args":{"sentiment":sent}}]

# ---- 4-step solvers (diagnostic) ----

def solve_driver(where, k):
    # L6: rank child aspects by negative share (severity), with a volume guard
    sh = add_shares(sentiment_breakdown(filter_data(CORPUS, **where), "child_aspect"))
    vol = apply_min_volume(sh, min_volume=5)
    top = rank_top(vol, by="neg_share", top_n=k)
    v = [[a, float(s)] for a, s in zip(top.child_aspect, top.neg_share.round(4))]
    return v, [
        {"tool":"sentiment_breakdown","args":{**where,"group_by":"child_aspect"}},
        {"tool":"add_shares","args":{}},
        {"tool":"apply_min_volume","args":{"min_volume":5}},
        {"tool":"rank_top","args":{"by":"neg_share","top_n":k}}]

def solve_prioritise(where, k):
    # L7: rank child aspects by priority = complaint volume x severity (prioritisation quadrant)
    sh = add_priority(add_shares(sentiment_breakdown(filter_data(CORPUS, **where), "child_aspect")))
    top = rank_top(sh, by="priority", top_n=k)
    v = list(zip(top.child_aspect, top.priority.round(4)))
    return v, [
        {"tool":"sentiment_breakdown","args":{**where,"group_by":"child_aspect"}},
        {"tool":"add_shares","args":{}},
        {"tool":"add_priority","args":{}},
        {"tool":"rank_top","args":{"by":"priority","top_n":k}}]

SOLVERS = {
    "solve_count": solve_count, "solve_share": solve_share,
    "solve_top_k": solve_top_k, "solve_split": solve_split,
    "solve_compare": solve_compare, "solve_two_prop": solve_two_prop,
    "solve_driver": solve_driver, "solve_prioritise": solve_prioritise,
} 

Task structure:
- tid: task ID, numbered by increasing difficulty
- type: question intent (descriptive, diagnostic or prescriptive)
- inds: industries
- solver: function that computes the gold answer 
- args: arguments to pass to that solver
- spec: structured description of the task, input to Gemini to write the question
- steps: number of tool calls in the gold path

## Create task specs

In [18]:
# generate task specs -- FULL taxonomy L1-L10, 2 tasks per level => 20 tasks.
#   industry-level L1-L5 (per_level each); org-level L6-L10 (one focus org per industry, so with
#   2 industries that is 2 tasks/level too). Reuses the Step 1 labels table.
#   L1 count/share, L2 split, L3 top_k, L4 compare, L5 two_prop  (descriptive/diagnostic);
#   L6 driver, L7 prioritise (diagnostic); L8 compare_diag, L9 recommend, L10 report (prescriptive).
#   Params are VARIED (driver/prioritise k; recommend/report industry-avg focus) so same-op
#   questions differ in substance, not just the subject. Prescriptive tasks compose `components`.
TASKS_PER_LEVEL = 2

def auto_tasks(industries, aspects, seed=42, per_level=TASKS_PER_LEVEL):
    rng = random.Random(seed)
    T = []

    def _add(tid, typ, inds, solver, args, spec, steps):
        T.append((tid, typ, set(inds), solver, args, spec, steps))

    def _comp(key, solver, args, op, steps):
        return {"key": key, "solver": solver, "args": args, "spec": {"op": op}, "steps": steps}

    def _compose(tid, ind, fo, op, components, spec_extra=None):
        steps = sum(c["steps"] for c in components)
        spec = {"op": op, "scope": "org", "subject": fo, "where": {"org": fo}, **(spec_extra or {})}
        _add(tid, "prescriptive", [ind], "solve_compose", (components,), spec, steps)

    pairs = list(itertools.combinations(industries, 2))
    sents = ["positive", "negative"]

    # ============ INDUSTRY-LEVEL: L1-L5 (per_level each) ============
    di = gi = 0

    # L1 count / share
    pool = [(ind, asp) for ind in industries for asp in aspects]
    rng.shuffle(pool)
    l1_ops = [("count", "negative"), ("share", "positive"), ("share", "negative")]
    for j in range(per_level):
        ind, asp = pool[j % len(pool)]
        op, sent = l1_ops[j % len(l1_ops)]
        di += 1
        _add(f"DESC-{di:02d}", "descriptive", [ind], f"solve_{op}", ({"industry": ind}, asp, sent),
             {"op": op, "scope": "industry", "subject": ind, "aspect": asp, "sent": sent}, 1)

    # L2 split
    for j in range(per_level):
        ind = industries[j % len(industries)]
        di += 1
        _add(f"DESC-{di:02d}", "descriptive", [ind], "solve_split", ({"industry": ind}, "industry"),
             {"op": "split", "scope": "industry", "subject": ind}, 2)

    # L3 top_k
    l3 = [("volume", 1), ("negative", 3), ("negative", 2)]
    for j in range(per_level):
        ind = industries[j % len(industries)]
        by, k = l3[j % len(l3)]
        # clamp k to the number of aspects that actually have rows of this kind
        pool_df = filter_data(CORPUS, industry=ind, **({"sentiment": "negative"} if by == "negative" else {}))
        n_avail = pool_df.child_aspect.nunique()
        k = min(k, n_avail)
        di += 1
        _add(f"DESC-{di:02d}", "descriptive", [ind], "solve_top_k", ({"industry": ind}, k, by),
             {"op": f"top_{by}", "scope": "industry", "subject": ind, **({"k": k} if k > 1 else {})}, 2)
        
    # L4 compare
    used = set()
    for j in range(per_level):
        pa, pb = pairs[j % len(pairs)]
        asp = rng.choice([a for a in aspects if a not in used] or aspects); used.add(asp)
        sent = sents[j % len(sents)]
        gi += 1
        _add(f"DIAG-{gi:02d}", "diagnostic", [pa, pb], "solve_compare",
             ({"industry": pa}, {"industry": pb}, asp, sent),
             {"op": "compare", "scope": "industry", "a": pa, "b": pb, "aspect": asp, "sent": sent}, 2)

    # L5 two_prop
    used = set()
    for j in range(per_level):
        pa, pb = pairs[j % len(pairs)]
        asp = rng.choice([a for a in aspects if a not in used] or aspects); used.add(asp)
        sent = rng.choice(sents)
        gi += 1
        _add(f"DIAG-{gi:02d}", "diagnostic", [pa, pb], "solve_two_prop",
             ({"industry": pa}, {"industry": pb}, asp, sent),
             {"op": "test", "scope": "industry", "a": pa, "b": pb, "aspect": asp, "sent": sent}, 3)

    # ============ ORG-LEVEL: L6-L10 (one focus org per industry -> 2 per level) ============
    # Fixed per-operation briefs: parameters do NOT vary across orgs (only the industry/org fills in).
    opi = 0                                                # gi continues from L4/L5 for DIAG ids
    for ind in industries:
        fo = ORGS[ind][0]                                  # focus org
        wfo, wind = {"org": fo}, {"industry": ind}
        # clamp k to the aspects available for this org (same as the L3 top-k clamp)
        n_asp = filter_data(CORPUS, org=fo).child_aspect.nunique()
        kd, kp = min(2, n_asp), min(3, n_asp)

        # L6 driver identification
        gi += 1
        _add(f"DIAG-{gi:02d}", "diagnostic", [ind], "solve_driver", (wfo, kd),
             {"op": "driver", "scope": "org", "subject": fo, "k": kd}, 4)
        # L7 prioritisation (volume x severity)
        gi += 1
        _add(f"DIAG-{gi:02d}", "diagnostic", [ind], "solve_prioritise", (wfo, kp),
             {"op": "prioritise", "scope": "org", "subject": fo, "k": kp}, 4)

        # L8 comparative diagnosis: org drivers vs the industry-average drivers
        opi += 1
        _compose(f"PRES-{opi:02d}", ind, fo, "compare_diag", [
            _comp("org_drivers", "solve_driver", (wfo, 3), "driver", 4),
            _comp("industry_avg_drivers", "solve_driver", (wind, 3), "driver", 4),
        ], {"industry": ind})
        # L9 targeted recommendation: overview + gap vs the industry average
        asp9, sent9 = rng.choice(aspects), rng.choice(sents)
        opi += 1
        _compose(f"PRES-{opi:02d}", ind, fo, "recommend", [
            _comp("overview", "solve_split", (wfo, "org"), "split", 2),
            _comp("vs_industry_avg", "solve_two_prop", (wfo, wind, asp9, sent9), "test", 3),
        ])
        # L10 full CX report: multiple descriptive + multiple diagnostic
        asp10, sent10 = rng.choice(aspects), rng.choice(sents)
        opi += 1
        _compose(f"PRES-{opi:02d}", ind, fo, "report", [
            _comp("overview", "solve_split", (wfo, "org"), "split", 2),
            _comp("top_complaints", "solve_top_k", (wfo, 3, "negative"), "top_negative", 2),
            _comp("drivers", "solve_driver", (wfo, 3), "driver", 4),
            _comp("priorities", "solve_prioritise", (wfo, 3), "prioritise", 4),
            _comp("vs_industry_avg", "solve_two_prop", (wfo, wind, asp10, sent10), "test", 3),
        ])

    return T

T = auto_tasks(INDUSTRIES, ASPECTS, seed=42)
n_ind = sum(t[5].get("scope") == "industry" for t in T)
n_org = sum(t[5].get("scope") == "org" for t in T)
print(f"{len(T)} tasks: L1-L10 x {TASKS_PER_LEVEL}/level -> {n_ind} industry (L1-L5) + {n_org} org (L6-L10)\n")
for tid, typ, inds, solver, args, spec, steps in T:
    print(f"{tid:10s} {steps:2d}-step  {spec['scope']:8s} {typ:12s}  {spec['op']:12s} subj={spec.get('subject') or spec.get('a')}  k={spec.get('k','-')}")

20 tasks: L1-L10 x 2/level -> 10 industry (L1-L5) + 10 org (L6-L10)

DESC-01     1-step  industry descriptive   count        subj=Consulting  k=-
DESC-02     1-step  industry descriptive   share        subj=Trading  k=-
DESC-03     2-step  industry descriptive   split        subj=Trading  k=-
DESC-04     2-step  industry descriptive   split        subj=Consulting  k=-
DESC-05     2-step  industry descriptive   top_volume   subj=Trading  k=-
DESC-06     2-step  industry descriptive   top_negative subj=Consulting  k=2
DIAG-01     2-step  industry diagnostic    compare      subj=Trading  k=-
DIAG-02     2-step  industry diagnostic    compare      subj=Trading  k=-
DIAG-03     3-step  industry diagnostic    test         subj=Trading  k=-
DIAG-04     3-step  industry diagnostic    test         subj=Trading  k=-
DIAG-05     4-step  org      diagnostic    driver       subj=TradeA  k=2
DIAG-06     4-step  org      diagnostic    prioritise   subj=TradeA  k=2
PRES-01     9-step  org      prescri

# Step 3: derive gold answers (JSON) and gold tool paths

In [19]:
# run solvers to get gold answers and paths (descriptive and diagnostic)
records = {}
for tid, type, inds, solver_name, args, spec, steps in T:
    if solver_name == "solve_compose":
        continue
    fn = SOLVERS[solver_name]
    ans, path = fn(*args)
    assert len(path) == steps, f"{tid}: expected {steps} steps, got {len(path)}"
    records[tid] = dict(task_id=tid, type=type, steps=steps,
                        industries=sorted(inds), spec=spec,
                        gold_answer=ans, gold_tool_path=path)

for tid in sorted(records)[:5]:
    print(f"{tid} ({records[tid]['steps']}-step): {records[tid]['gold_answer']}")

DESC-01 (1-step): 9
DESC-02 (1-step): 0.071
DESC-03 (2-step): {'pos_share': 0.432, 'neg_share': 0.523, 'neu_share': 0.045}
DESC-04 (2-step): {'pos_share': 0.5, 'neg_share': 0.45, 'neu_share': 0.05}
DESC-05 (2-step): ['app-website']


In [20]:
# build prescriptive tasks (L8 comparative diagnosis, L9 recommendation, L10 full report).
# run each component analysis and concatenate their tool paths.
# gold_answer = {"findings": {component_key: answer}}.
for tid, type, inds, solver_name, args, spec, steps in T:
    if solver_name != "solve_compose":
        continue
    components = args[0]
    path, findings, meta = [], {}, []
    for c in components:
        ans, cpath = SOLVERS[c["solver"]](*c["args"])
        assert len(cpath) == c["steps"], f"{tid}/{c['key']}: {len(cpath)} != {c['steps']}"
        findings[c["key"]] = ans
        path += cpath
        meta.append({"key": c["key"], "spec": c["spec"], "steps": c["steps"]})
    assert len(path) == steps, f"{tid}: {len(path)} != {steps}"
    records[tid] = dict(task_id=tid, type=type, steps=steps, industries=sorted(inds),
                        spec={**spec, "components": meta},
                        gold_answer={"findings": findings}, gold_tool_path=path)
    print(f"{tid} ({steps}-step, {spec['op']}): {[c['key'] for c in components]}")

PRES-01 (9-step, compare_diag): ['org_drivers', 'industry_avg_drivers'] + sample_reviews
PRES-02 (6-step, recommend): ['overview', 'vs_industry_avg'] + sample_reviews
PRES-03 (16-step, report): ['overview', 'top_complaints', 'drivers', 'priorities', 'vs_industry_avg'] + sample_reviews
PRES-04 (9-step, compare_diag): ['org_drivers', 'industry_avg_drivers'] + sample_reviews
PRES-05 (6-step, recommend): ['overview', 'vs_industry_avg'] + sample_reviews
PRES-06 (16-step, report): ['overview', 'top_complaints', 'drivers', 'priorities', 'vs_industry_avg'] + sample_reviews


## Verify gold tool paths and answers

Sanity check: independently **re-run each recorded `gold_tool_path`** against `CORPUS`
(calling the real tool functions directly, *not* the solver functions that produced them) and
confirm it reproduces `gold_answer`. A wrong tool name or argument in a path would make the two
diverge. The final answer is read off the last tool's output, shaped per the task's `spec["op"]`.

In [21]:
# flag tasks whose gold answer may be fragile or mismatched (run before saving tasks.csv)
issues = []
for tid, r in records.items():
    gj, spec = r["gold_answer"], r["spec"]
    # top-k asked for more than exists
    if spec.get("op", "").startswith(("top_", "driver", "prioritise")):
        k = spec.get("k")
        if isinstance(gj, list) and k and len(gj) < k:
            issues.append(f"{tid}: asked {k}, only {len(gj)} available")
    # tied scores -> ranking order arbitrary
    if isinstance(gj, list) and gj and isinstance(gj[0], (list, tuple)) and len(gj[0]) == 2:
        scores = [s for _, s in gj]
        if len(scores) != len(set(scores)):
            issues.append(f"{tid}: tied scores {scores} -> order not well-defined")
    # very low/high share on a thin cell
    if spec.get("op") == "share" and isinstance(gj, (int, float)) and (gj < 0.1 or gj > 0.95):
        issues.append(f"{tid}: extreme share {gj} -> check cell size / question wording")

print(f"{len(issues)} issue(s):")
for i in issues: print("  -", i)

2 issue(s):
  - DESC-02: extreme share 0.071 -> check cell size / question wording
  - DIAG-06: tied scores [1.0, 1.0] -> order not well-defined


In [22]:
# Re-run every recorded gold_tool_path against the in-memory CORPUS with the REAL tool functions
# (not the solvers that produced them) and check it reproduces gold_answer. A wrong tool name or
# argument would make the two diverge.
# We verify the `records` generated just above (Steps 2-3), NOT a reloaded tasks.csv: this cell
# runs before Step 5 writes the CSV, so loading that file would pick up a PRIOR run's tasks
# (possibly with retired tool names). `question` is added in Step 4, so it's optional here.
#   count -> share_of[...]["count"]   share -> share_of[...]["share"]
#   prescriptive (recommend/compare_diag/report): findings read by slicing the path per component.
import inspect
import tools as _tools_mod

# guard: every tool named in a gold_tool_path must be a real tool defined in the tools module.
REAL_TOOLS = {n for n, f in inspect.getmembers(_tools_mod, inspect.isfunction)
              if f.__module__ == _tools_mod.__name__ and not n.startswith("_")}
for _tid, _rec in records.items():
    for _step in _rec["gold_tool_path"]:
        assert _step["tool"] in REAL_TOOLS, (
            f"{_tid}: gold_tool_path references {_step['tool']!r}, which is not a real tool in the "
            f"tools module. Real tools: {sorted(REAL_TOOLS)}")
print(f"tool-name check passed: every path uses only real tools ({len(REAL_TOOLS)} available)")

def _slice(args, keys):
    return filter_data(CORPUS, **{k: args[k] for k in keys if k in args})

# key sets exclude 'sentiment' where a tool must NOT pre-filter by sentiment (share_of, breakdown)
_SEG = ("industry", "org", "child_aspect")               # segment slice (no sentiment)
_SEG_S = ("industry", "org", "child_aspect", "sentiment")  # segment slice incl. sentiment
_COMPOSE_OPS = ("recommend", "compare_diag", "report")

def run_gold_path(path, spec, records):
    op = spec["op"]

    # prescriptive: findings = each component's answer, read by slicing the path per component
    if op in _COMPOSE_OPS:
        findings, idx = {}, 0
        for comp in spec["components"]:
            n = comp["steps"]
            findings[comp["key"]] = run_gold_path(path[idx:idx + n], comp["spec"], records)
            idx += n
        return {"findings": findings}

    cur = None                            # current DataFrame for chained ops
    share_slices, share_results = [], []  # from share_of steps -> feed count/share/compare/two_prop
    tp = None
    for step in path:
        t, a = step["tool"], step["args"]
        if t == "share_of":
            df = _slice(a, _SEG)
            share_slices.append(df)
            share_results.append(share_of(df, a["sentiment"]))
        elif t == "count_by":
            cur = count_by(_slice(a, _SEG_S + ("parent_aspect",)), a["group_by"])
        elif t == "sentiment_breakdown":
            cur = sentiment_breakdown(_slice(a, _SEG), a["group_by"])
        elif t == "add_shares":
            cur = add_shares(cur)
        elif t == "add_priority":
            cur = add_priority(cur)
        elif t == "apply_min_volume":
            cur = apply_min_volume(cur, min_volume=a.get("min_volume", 30))
        elif t == "rank_top":
            cur = rank_top(cur, by=a["by"], top_n=a["top_n"])
        elif t == "two_prop_test":
            tp = two_prop_test(share_slices[-2], share_slices[-1], a["sentiment"])
        else:
            raise ValueError(f"unknown tool in path: {t}")

    # read the final answer off the last tool's output, shaped per task op
    if op == "count":
        return share_results[-1]["count"]
    if op == "share":
        return share_results[-1]["share"]
    if op in ("top_volume", "top_negative"):
        return cur["child_aspect"].tolist()
    if op == "split":
        return {k: float(cur[k].iloc[0]) for k in ("pos_share", "neg_share", "neu_share")}
    if op == "compare":
        a_, b_ = share_results[0]["share"], share_results[1]["share"]
        return {"a": a_, "b": b_, "higher": "a" if (a_ or 0) > (b_ or 0) else "b"}
    if op == "test":
        return tp
    if op == "driver":
        return [[asp, float(round(ns, 4))] for asp, ns in zip(cur["child_aspect"], cur["neg_share"])]
    if op == "prioritise":
        return [[asp, float(round(p, 4))] for asp, p in zip(cur["child_aspect"], cur["priority"])]
    raise ValueError(f"unknown op: {op}")


def _match(a, b, tol=1e-4):
    """Tolerant structural comparison: list==tuple, floats within tol, exact otherwise."""
    if isinstance(a, bool) or isinstance(b, bool):
        return a == b
    if a is None or b is None:
        return a is None and b is None
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        return math.isclose(float(a), float(b), abs_tol=tol)
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_match(a[k], b[k], tol) for k in a)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_match(x, y, tol) for x, y in zip(a, b))
    return a == b


print(f"Verifying {len(records)} gold tool paths against CORPUS ({len(CORPUS)} reviews)")
print("=" * 78)
n_pass = 0
for tid, rec in records.items():
    got = run_gold_path(rec["gold_tool_path"], rec["spec"], records)
    ok = _match(got, rec["gold_answer"])
    n_pass += ok
    print(f"\n[{tid}] {rec['steps']}-step  {rec.get('question', '(question written in Step 4)')}")
    print(f"  gold answer : {rec['gold_answer']}")
    print(f"  path re-ran : {got}")
    print(f"  {'PASS' if ok else 'FAIL <<<<<<'}")

print("\n" + "=" * 78)
print(f"{n_pass}/{len(records)} tasks: gold_tool_path reproduces gold_answer")

tool-name check passed: every path uses only real tools (12 available)
Verifying 20 gold tool paths against CORPUS (104 reviews)

[DESC-01] 1-step  (question written in Step 5)
  gold answer : 9
  path re-ran : 9
  PASS

[DESC-02] 1-step  (question written in Step 5)
  gold answer : 0.071
  path re-ran : 0.071
  PASS

[DESC-03] 2-step  (question written in Step 5)
  gold answer : {'pos_share': 0.432, 'neg_share': 0.523, 'neu_share': 0.045}
  path re-ran : {'pos_share': 0.432, 'neg_share': 0.523, 'neu_share': 0.045}
  PASS

[DESC-04] 2-step  (question written in Step 5)
  gold answer : {'pos_share': 0.5, 'neg_share': 0.45, 'neu_share': 0.05}
  path re-ran : {'pos_share': 0.5, 'neg_share': 0.45, 'neu_share': 0.05}
  PASS

[DESC-05] 2-step  (question written in Step 5)
  gold answer : ['app-website']
  path re-ran : ['app-website']
  PASS

[DESC-06] 2-step  (question written in Step 5)
  gold answer : ['competitor', 'app-website']
  path re-ran : ['competitor', 'app-website']
  PASS

[DIA

# Step 4: generate task questions

In [23]:
# use LLM to write questions from task specs
start = time.time()
def brief(sp):
    o = sp["op"]; who = sp.get("subject")
    if o == "count":        return f"How many complaints {who} received about {sp['aspect']}."
    if o == "share":        return f"The share of {who} feedback on {sp['aspect']} that is {'praise' if sp['sent']=='positive' else 'complaints'}."
    if o == "top_volume":   return f"Which single topic {who} customers mention most."
    if o == "top_negative": return f"The {sp['k']} topics with the most complaints for {who}."
    if o == "split":        return f"The overall breakdown of happy vs unhappy {who} customers."
    if o == "compare":      return f"Whether {sp['a']} or {sp['b']} customers have more {'complaints about' if sp['sent']=='negative' else 'praise for'} {sp['aspect']}."
    if o == "test":         return f"Whether the difference in {'complaints about' if sp['sent']=='negative' else 'praise for'} {sp['aspect']} between {sp['a']} and {sp['b']} is statistically significant."
    if o == "driver":       return (f"The single biggest driver of dissatisfaction for {who}." if sp['k']==1
                                    else f"The top {sp['k']} drivers of dissatisfaction for {who}.")
    if o == "prioritise":   return f"The top {sp['k']} issues {who} should fix first, weighing both how many customers are affected and how unhappy they are."
    if o == "compare_diag": return f"How {who}'s biggest complaint drivers compare with the {sp['industry']} industry average, and where {who} does worse."
    if o == "recommend":    return f"Diagnose the biggest customer problems for {who} and recommend the top fixes."
    if o == "report":       return f"A full customer-experience report for {who}: overall happiness, top complaints, biggest drivers, what to prioritise, and how it compares to the industry average."

briefs = {tid: brief(r["spec"]) for tid, r in records.items()}

qs = gemini_json(
 "You are a CX analytics lead writing questions for a business intelligence tool.\n\n"
  "For each item below, write ONE clear, specific, realistic question that a business "
  "stakeholder would ask. The question must be answerable using exactly the "
  "analysis described in the item.\n\n"

  "LANGUAGE:\n"
  "- Use professional CX vocabulary: satisfaction, sentiment, opinion, feedback, "
  "complaints, issues, problems, praise, drivers.\n"
  "- Do NOT use casual words like 'happy', 'unhappy', 'like', or 'love'. Instead say "
  "'satisfaction', 'dissatisfaction', 'positive sentiment', 'negative sentiment', or "
  "'complaints'.\n"
  "- Never write phrases like 'level of unhappiness' or 'degree of happiness'. Use "
  "'dissatisfaction' or 'negative sentiment'.\n"
  "- Keep every question under 20 words. Prefer one clause. Cut filler like "
  "'Could you show me', 'I'd like to see', 'Please provide'.\n\n"

  "TRANSLATE TOPICS INTO NATURAL PHRASING:\n"
  "- 'app-website'          -> 'the app or website'\n"
  "- 'ease-of-use'          -> 'ease of use'\n"
  "- 'account-access'       -> 'account access'\n"
  "- 'price-value-for-money'-> 'value for money'\n"
  "(Apply the same natural-language treatment to any other topic.)\n\n"

  "NAMES AND COMPARISONS:\n"
  "- Some items refer to a whole industry (e.g. 'Banking'); others to a specific company "
  "(e.g. 'Halden Savings', 'Corvex IT'). Always keep the exact name given.\n"
  "- When comparing one company against the rest of its industry, use the phrase "
  "'the industry average'.\n"
  "- Preserve any specific counts, such as 'top 3'.\n\n"

  "OUTPUT:\n"
  "Return ONLY a JSON object mapping each item's id to its question string. Use the ids "
  "EXACTLY as given, and include every id.\n\n"
  + json.dumps(briefs), block="questions", max_tokens=6000)  # one question per task; give the
  # same headroom as the answers call so a large batch's JSON isn't truncated (-> parse failure)

# map each generated question back onto its task by id. Iterate over `records` (not qs) so a
# stray/renamed id from the model can't KeyError here, and surface any task it skipped -- an
# unfilled `question` would otherwise blow up downstream in Step 5.
missing = [tid for tid in records if tid not in qs]
if missing:
    raise ValueError(f"question generation returned no question for {len(missing)} task(s): {missing}")
for tid in records:
    records[tid]["question"] = qs[tid]
    print(f"{tid}: {qs[tid]}")

elapsed = time.time() - start
print(f"\nquestions generated in {elapsed:.1f}s")
cost_report()

DESC-01: How many complaints did Consulting receive about the app or website?
DESC-02: What share of Trading feedback on competitor is praise?
DESC-03: What is the overall satisfaction breakdown for Trading customers?
DESC-04: What is the overall satisfaction breakdown for Consulting customers?
DESC-05: Which feedback topic do Trading customers mention most frequently?
DESC-06: What are the top 2 complaint topics for Consulting?
DIAG-01: Which customers, Trading or Consulting, show more praise for the app or website?
DIAG-02: Which customers, Trading or Consulting, have more complaints about competitor?
DIAG-03: Is the praise difference for the app or website between Trading and Consulting statistically significant?
DIAG-04: Is the complaint difference about competitor between Trading and Consulting statistically significant?
DIAG-05: What are the top 2 drivers of dissatisfaction for TradeA?
DIAG-06: What are TradeA's top 2 priority issues based on dissatisfaction impact?
DIAG-07: What

# Step 5: generate gold answers (text)

In [24]:
# # only needed after restarting kernel
# # Load the saved tasks export and rebuild the minimal structures expected by Step 5.
# # This lets the question-generation and answer-generation cells run without executing the earlier cells.

# tasks = pd.read_csv(VER_DIR / MODE / "tasks.csv")

# # Rebuild the lightweight records structure expected by the notebook.
# records = {}
# for _, row in tasks.iterrows():
#     tid = row["task_id"]
#     records[tid] = {
#         "task_id": tid,
#         "question": row.get("question", ""),
#         "gold_answer": row.get("gold_answer"),
#         "gold_tool_path": row.get("gold_tool_path"),
#     }

# # Rebuild a minimal T structure so the final save cell can still write tasks.csv in the same format.
# # The notebook later uses T to preserve task order and the task_id list.
# T = [(row["task_id"], None, None, None, None, None, None) for _, row in tasks.iterrows()]


In [25]:
# business-facing natural-language gold answer (generated AFTER the JSON answer + question).
# Each NL answer must use ONLY the figures/insights in the JSON gold answer and answer that task's
# question directly. Stored as records[tid]["gold_answer_nl"] -> a second gold-answer column.
start = time.time()
def _sides(rec):
    """Resolve the 'a'/'b' placeholders in a comparison gold answer to real entity names, so the
    model can't invert 'which side is higher'. Only compare/test (flat) and the org-vs-industry
    'vs_industry_avg' finding in recommend/report use a/b; each task has at most one such compare."""
    sp = rec["spec"]; op = sp["op"]
    if op in ("compare", "test"):                       # a and b are the two industries compared
        return {"a": sp["a"], "b": sp["b"]}
    if op in ("recommend", "report"):                   # a = the focus org, b = its industry average
        ind = rec["industries"][0] if rec.get("industries") else "the"
        return {"a": sp["subject"], "b": f"the {ind} industry average"}
    return {}


def _ties_in(gold):
    """Detect tied rankings so the NL answer states the tie explicitly instead of an arbitrary order.
    Scans [[label, score], ...] ranking lists (driver/prioritise, incl. inside a report's findings)
    and reports any group of labels that share the same score."""
    from collections import defaultdict
    notes = []
    def is_rank(x):
        return (isinstance(x, list) and len(x) >= 2 and
                all(isinstance(e, (list, tuple)) and len(e) == 2
                    and isinstance(e[1], (int, float)) and not isinstance(e[1], bool) for e in x))
    def scan(x, ctx=""):
        if is_rank(x):
            g = defaultdict(list)
            for lab, sc in x:
                g[round(float(sc), 4)].append(lab)
            for sc, labs in g.items():
                if len(labs) > 1:
                    notes.append(f"{', '.join(labs)} are tied at {sc}" + (f" (under {ctx})" if ctx else ""))
        elif isinstance(x, dict):
            for kk, vv in x.items():
                scan(vv, "" if kk == "findings" else kk)
    scan(gold)
    return notes


nl_inputs = {}
for tid, r in records.items():
    item = {"question": r["question"], "gold_answer": r["gold_answer"]}
    sides = _sides(r)                                   # only present for tasks with an a/b comparison
    if sides:
        item["sides"] = sides
    ties = _ties_in(r["gold_answer"])                   # tied ranks -> the answer must call out the tie
    if ties:
        item["ties"] = ties
    nl_inputs[tid] = item

nl = gemini_json(
  "You are a CX analytics lead writing the answer a stakeholder receives.\n\n"
  "For each item you are given the stakeholder's QUESTION and the correct ANSWER as "
  "structured JSON (the ground truth). Write ONE clear, business-facing answer that "
  "directly answers the question.\n\n"

  "MOST IMPORTANT RULE:\n"
  "Use ONLY the figures and findings in the JSON answer. Never invent numbers, add "
  "detail, or state anything the JSON does not support. If it isn't in the JSON, it "
  "does not go in the answer.\n\n"

"LANGUAGE:\n"
  "- Plain, professional business English using CX vocabulary: satisfaction, "
  "dissatisfaction, sentiment, feedback, complaints, issues, praise, drivers.\n"
  "- Do NOT use casual words like 'happy', 'unhappy', or 'like'. For a sentiment split, "
  "write e.g. '43% positive, 52% negative, 5% neutral' rather than 'happy/unhappy'.\n"
  "- Dont restate the question.\n\n"

  "NUMBERS:\n"
  "- Render proportions/shares as whole-number percentages (0.725 -> '73%').\n"
  "- Keep counts as whole numbers.\n"
  "- A 'priority' value (in prioritise / 'priorities' findings) is a 0-2 ranking score, NOT a "
  "proportion: list those items in ranked order and do NOT render the score as a percentage.\n\n"

  "TRANSLATE TOPICS INTO NATURAL PHRASING:\n"
  "- 'app-website'           -> 'the app or website'\n"
  "- 'ease-of-use'           -> 'how easy the service is to use'\n"
  "- 'account-access'        -> 'signing in or accessing their account'\n"
  "- 'attitude-of-staff'     -> 'staff attitude'\n"
  "- 'price-value-for-money' -> 'value for money'\n"
  "- 'discounts-promotions'  -> 'discounts and promotions'\n\n"

  "NAMES AND COMPARISONS:\n"
  "- Use the exact company and industry names from the JSON.\n"
  "- Comparison answers label the two sides 'a' and 'b' (e.g. 'higher': 'a'; in a significance "
  "test 'p1' is side a's rate and 'p2' is side b's). A 'sides' object maps 'a' and 'b' to the real "
  "entities -- state the result using those names and take the direction ONLY from 'higher'; never "
  "guess which side is higher.\n"
  "- In a 'vs_industry_avg' finding, side 'a' is the company and side 'b' is the industry average; "
  "use the phrase 'the industry average' for side b.\n\n"

  "TIES (IMPORTANT):\n"
  "- A 'ties' list, when present, names ranked items that share the SAME score. Do NOT order them "
  "or pick one over the others: say plainly that they are tied and back it up with the shared "
  "figure for each (apply the NUMBERS rules above when rendering it).\n\n"

  "ANSWER SHAPE BY TASK TYPE:\n"
  "- Significance test: state whether the difference is statistically significant "
  "(p < 0.05) and which side is higher.\n"
  "- Report or recommendation: synthesise the findings into a short paragraph that "
  "ends with the single most important action to take.\n"
  "- Everything else: 1-2 sentences that answer the question directly.\n\n"

  "FORMATTING (IMPORTANT):\n"
  "- Do NOT use any quotation marks inside your answer text — no double quotes and no "
  "single quotes. Refer to features in plain words without quoting them.\n"  # quotes cause errors in JSON parsing
  "- Write plain sentences only: no markdown, no line breaks inside an answer.\n\n"

  "OUTPUT:\n"
  "Return ONLY a JSON object mapping each item's id to its answer string. Use the ids EXACTLY "
  "as given, and include every id.\n\n"
  + json.dumps(nl_inputs, default=str), block="answers", max_tokens=16000)

# map each answer back onto its task by id. Iterate over `records` (not nl) so a stray/renamed id
# from the model can't KeyError here, and surface any task it skipped (an unfilled gold_answer_nl
# would otherwise be written to tasks.csv as an empty answer column in the next cell).
missing = [tid for tid in records if tid not in nl]
if missing:
    raise ValueError(f"answer generation returned no answer for {len(missing)} task(s): {missing}")
for tid in records:
    records[tid]["gold_answer_nl"] = nl[tid]
    print(f"{tid}: {nl[tid]}")

elapsed = time.time() - start
print(f"\nnatural-language answers generated in {elapsed:.1f}s")

DESC-01: Consulting received 9 complaints about the app or website.
DESC-02: 7% of Trading feedback regarding competitor is praise.
DESC-03: Overall satisfaction for Trading customers is 43% positive, 52% negative, and 5% neutral.
DESC-04: Overall satisfaction for Consulting customers is 50% positive, 45% negative, and 5% neutral.
DESC-05: Trading customers most frequently mention the app or website.
DESC-06: The top two complaint topics for Consulting are competitor and the app or website.
DIAG-01: Consulting customers show more praise for the app or website.
DIAG-02: Trading customers have more complaints about competitor.
DIAG-03: The difference in praise for the app or website between Trading and Consulting is not statistically significant. Consulting shows higher praise.
DIAG-04: The difference in complaints about competitor between Trading and Consulting is not statistically significant. Trading shows higher complaints.
DIAG-05: The top two drivers of dissatisfaction for TradeA a

In [26]:
# save tasks.csv and print cost report. Tasks are written beside the dataset they were built
# against: benchmark_outputs/{VERSION}/{MODE}/ holds labels.csv + cells.csv (Step 1) and tasks.csv (here).
def _clean(r):
    """Make record JSON-serialisable (sets -> sorted lists)."""
    r = dict(r)
    if isinstance(r.get("industries"), set):
        r["industries"] = sorted(r["industries"])
    return r

order = list(records)   # task ids in generation order (records is built from T); no need to depend on T
tasks_csv_path = VER_DIR / MODE / "tasks.csv"

# two gold-answer columns -> gold_answer_json (valid JSON, for deterministic checks) and
# gold_answer_text (business-facing natural language). spec / gold_tool_path stay Python-repr.
# operation -> taxonomy level (L1-L10); each op maps to exactly one level in this taxonomy.
OP_LEVEL = {"count": 1, "share": 1, "split": 2, "top_volume": 3, "top_negative": 3,
            "compare": 4, "test": 5, "driver": 6, "prioritise": 7,
            "compare_diag": 8, "recommend": 9, "report": 10}

def _csv_row(tid):
    r = _clean(records[tid])
    r["mode"] = MODE                               # which Step 1 dataset this was built on
    sp = r["spec"]                                 # break spec fields out into their own columns
    r["level"] = OP_LEVEL.get(sp.get("op"))        # taxonomy level (6-10 for the org-level set)
    r["scope"] = sp.get("scope")                   # 'industry' or 'org'
    r["operation"] = sp.get("op")                  # e.g. count, share, driver, report
    r["industry_or_org"] = sp.get("subject")       # the industry or org the task is about
    r["gold_answer_json"] = json.dumps(r.pop("gold_answer"))
    r["gold_answer_text"] = r.pop("gold_answer_nl", None)
    return r

tasks_df = pd.DataFrame([_csv_row(tid) for tid in order])
col_order = ["task_id", "mode", "level", "type", "steps", "scope", "operation", "industry_or_org",
             "industries", "spec", "question",
             "gold_answer_json", "gold_answer_text", "gold_tool_path"]
tasks_df = tasks_df[[c for c in col_order if c in tasks_df.columns]
                    + [c for c in tasks_df.columns if c not in col_order]]
tasks_df.to_csv(tasks_csv_path, index=False)

print(f"wrote {tasks_csv_path}")
print(f"(dataset: {VER_DIR / MODE / 'labels.csv'})\n")
cost_report()

wrote ..\benchmark_outputs\2026-07-23_080721_tasks.csv
(corpus: ..\benchmark_outputs\2026-07-23_080721_corpus.csv | insights: ..\benchmark_outputs\2026-07-23_080721_insights.csv)

questions    calls=  1  in=    865  out=   2635  $0.0068
answers      calls=  1  in=   2578  out=   6184  $0.0162
TOTAL                               $0.0231


In [27]:
tasks_df.head(3)

,task_id,level,type,steps,scope,operation,industry_or_org,industries,spec,question,gold_answer_json,gold_answer_text,gold_tool_path
0,DESC-01,1,descriptive,1,industry,count,Consulting,[Consulting],"{'op': 'count', 'scope': 'industry', 'subject': 'Consulting', 'aspect': 'app-website', 'sent': 'negative'}",How many complaints did Consulting receive about the app or website?,9,Consulting received 9 complaints about the app or website.,"[{'tool': 'share_of', 'args': {'industry': 'Consulting', 'child_aspect': 'app-website', 'sentiment': 'negative'}}]"
1,DESC-02,1,descriptive,1,industry,share,Trading,[Trading],"{'op': 'share', 'scope': 'industry', 'subject': 'Trading', 'aspect': 'competitor', 'sent': 'positive'}",What share of Trading feedback on competitor is praise?,0.071,7% of Trading feedback regarding competitor is praise.,"[{'tool': 'share_of', 'args': {'industry': 'Trading', 'child_aspect': 'competitor', 'sentiment': 'positive'}}]"
2,DESC-03,2,descriptive,2,industry,split,Trading,[Trading],"{'op': 'split', 'scope': 'industry', 'subject': 'Trading'}",What is the overall satisfaction breakdown for Trading customers?,"{""pos_share"": 0.432, ""neg_share"": 0.523, ""neu_share"": 0.045}","Overall satisfaction for Trading customers is 43% positive, 52% negative, and 5% neutral.","[{'tool': 'sentiment_breakdown', 'args': {'industry': 'Trading', 'group_by': 'industry'}}, {'tool': 'add_shares', 'args': {}}]"


# End of notebook